In [1]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts

In [2]:
df = pl.read_parquet("../final_patient_data.parquet")

In [3]:
df.head(1)

patient_id,hosp_id,centre,hosp_admission_datetime,hosp_discharge_datetime,inclusion_flag,icu_stay_flag,Cardiopathie ischémique,Fibrillation atriale,Insuffisance cardiaque chronique,Pacemaker,Pontage aorto-coronarien,Insuffisance rénale chronique,Antécédent d'AVC,Cirrhose,Cancer actif,Immunodépression,Dyslipidémie,Diabète,beta_blocker,ACEI,ARB,anti_aldosterone,furosemide,gender,birth_date,death_date,age_at_admission,comorbidity_score,hfrs_score,OMT_component_score,death_flag
str,i64,str,datetime[μs],datetime[μs],i64,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,str,date,date,i32,f64,f64,i8,i32
"""P001""",1002,"""angers""",2018-07-04 11:15:00,2018-07-12 10:00:00,1,0,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,"""F""",1933-04-12,2018-09-21,85,0.0,0.0,2,1


In [4]:
cv_cols = [
    "Cardiopathie ischémique",
    "Fibrillation atriale",
    "Insuffisance cardiaque chronique",
    "Pacemaker",
    "Pontage aorto-coronarien",
    "Antécédent d'AVC",
]
df = df.with_columns(
    (
        pl.sum_horizontal([pl.col(c) for c in cv_cols]) >= 1
    )
    .cast(pl.Int8)
    .alias("cv_history")
)

In [5]:
df = df.select([
    "patient_id", 
    "hosp_admission_datetime", 
    "death_date", 
    "cv_history", 
    "gender", 
    "age_at_admission",
    "comorbidity_score", 
    "hfrs_score",
    "furosemide",
    "OMT_component_score"
])

In [6]:
df.head(1)

patient_id,hosp_admission_datetime,death_date,cv_history,gender,age_at_admission,comorbidity_score,hfrs_score,furosemide,OMT_component_score
str,datetime[μs],date,i8,str,i32,f64,f64,i8,i8
"""P001""",2018-07-04 11:15:00,2018-09-21,1,"""F""",85,0.0,0.0,0,2


In [7]:
df = df.to_pandas()

In [8]:
# Convert to datetime and then extract the date only
# Normalize the time component to 00:00:00
df["hosp_admission_datetime"] = pd.to_datetime(df["hosp_admission_datetime"]).dt.normalize()
df["death_date"] = pd.to_datetime(df["death_date"]).dt.normalize()

In [9]:
# Create a one-year follow-up limit for each patient (admission date + 365 days)
df["one_year_limit"] = df["hosp_admission_datetime"]+pd.Timedelta(days=365)

In [10]:
df.head(1)

,patient_id,hosp_admission_datetime,death_date,cv_history,gender,age_at_admission,comorbidity_score,hfrs_score,furosemide,OMT_component_score,one_year_limit
0,P001,2018-07-04,2018-09-21,1,F,85,0.0,0.0,0,2,2019-07-04


In [11]:
# Define the actual end of follow-up for each patient:
# use the death date if available; otherwise, use the one-year limit
df["end_followup"] = df["death_date"].fillna(df["one_year_limit"])

# If the death date is beyond the one-year limit, censor follow-up at the one-year limit
df.loc[df["end_followup"] > df["one_year_limit"], "end_followup"] = (df["one_year_limit"])

In [12]:
# 3. Calculate the elapsed follow-up time in days
# This is the time variable used in the Kaplan-Meier estimator
df["time"] = (
    df["end_followup"]-df["hosp_admission_datetime"]
).dt.days

In [13]:
# 4. Determine whether death occurred within this follow-up period
# This is the status/event variable
df["status"] = (df["death_date"].notna()) & (
    df["death_date"] < df["one_year_limit"]
)

In [14]:
df.head()

,patient_id,hosp_admission_datetime,death_date,cv_history,gender,age_at_admission,comorbidity_score,hfrs_score,furosemide,OMT_component_score,one_year_limit,end_followup,time,status
0,P001,2018-07-04,2018-09-21,1,F,85,0.0,0.0,0,2,2019-07-04,2018-09-21,79,True
1,P002,2019-01-18,NaT,0,M,84,0.0,0.0,0,0,2020-01-18,2020-01-18,365,False
2,P003,2019-04-09,2020-12-04,1,F,83,3.0,1.4,1,2,2020-04-08,2020-04-08,365,False
3,P004,2018-03-03,NaT,1,M,82,0.0,0.0,0,1,2019-03-03,2019-03-03,365,False
4,P005,2019-02-01,2019-03-17,0,F,81,2.0,0.0,0,0,2020-02-01,2019-03-17,44,True


In [15]:
df_surv = df.loc[df['cv_history']==1, [
    'patient_id',  
    'time',
    'status', 
    'cv_history',
    'gender',
    'age_at_admission',
    'comorbidity_score',
    'hfrs_score',
    "furosemide",
    'OMT_component_score']].copy()

In [16]:
df_surv.head()

,patient_id,time,status,cv_history,gender,age_at_admission,comorbidity_score,hfrs_score,furosemide,OMT_component_score
0,P001,79,True,1,F,85,0.0,0.0,0,2
2,P003,365,False,1,F,83,3.0,1.4,1,2
3,P004,365,False,1,M,82,0.0,0.0,0,1
7,P008,209,True,1,M,80,0.0,3.7,0,0
9,P010,21,True,1,M,83,0.0,0.0,0,0


In [17]:
df_surv[df_surv['time']<0]

,patient_id,time,status,cv_history,gender,age_at_admission,comorbidity_score,hfrs_score,furosemide,OMT_component_score


In [18]:
df_surv = df_surv[df_surv['time']>= 0]

In [19]:
df_surv.isna().sum()

patient_id             0
time                   0
status                 0
cv_history             0
gender                 0
age_at_admission       0
comorbidity_score      0
hfrs_score             0
furosemide             0
OMT_component_score    0
dtype: int64

In [20]:
df_surv.head(1)

,patient_id,time,status,cv_history,gender,age_at_admission,comorbidity_score,hfrs_score,furosemide,OMT_component_score
0,P001,79,True,1,F,85,0.0,0.0,0,2


In [21]:
df_surv["status"] = df_surv["status"].astype(int)

In [22]:
df_surv["gender"] = (df_surv["gender"].replace({
    "M": 1,
    "F": 0
    })
    .astype("Int64")                
)

C:\Users\CICP\AppData\Local\Temp\ipykernel_8824\80980484.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_surv["gender"] = (df_surv["gender"].replace({


In [23]:
df_surv.head(1)

,patient_id,time,status,cv_history,gender,age_at_admission,comorbidity_score,hfrs_score,furosemide,OMT_component_score
0,P001,79,1,1,0,85,0.0,0.0,0,2


In [24]:
from lifelines import CoxPHFitter

cph_omt_furo = CoxPHFitter()

cph_omt_furo.fit(
    df_surv,
    duration_col = "time",
    event_col = "status",
    formula = "OMT_component_score*furosemide"
)

cph_omt_furo.print_summary

C:\Users\CICP\anaconda3\envs\arefe_genially\Lib\site-packages\lifelines\fitters\coxph_fitter.py:1530: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.905921325808308e-17.
  inv_h_dot_g_T = spsolve(-h, g, assume_a="pos", check_finite=False)
C:\Users\CICP\anaconda3\envs\arefe_genially\Lib\site-packages\lifelines\utils\__init__.py:1120: ConvergenceWarning: Column furosemide have very low variance when conditioned on death event present or not. This may harm convergence. This could be a form of 'complete separation'. For example, try the following code:

>>> events = df['status'].astype(bool)
>>> print(df.loc[events, 'furosemide'].var())
>>> print(df.loc[~events, 'furosemide'].var())

A very low variance means that the column furosemide completely determines whether a subject dies or not. See https://stats.stackexchange.com/questions/11109/how-to-deal-with-perfect-separation-in-logistic-regression.

  warnings.warn(dedent(warning_text), ConvergenceWarning)


ConvergenceError: Convergence halted due to matrix inversion problems. Suspicion is high collinearity. Please see the following tips in the lifelines documentation: https://lifelines.readthedocs.io/en/latest/Examples.html#problems-with-convergence-in-the-cox-proportional-hazard-modelA singular matrix detected: slice(s) [0] are singular.